# Escolher uma imagem e gerar uma paleta

Este notebook é uma ferramenta independente para testar a Parte 1. Não precisa executar os notebooks do trabalho e não altera seus resultados.

**Como usar:**

1. Execute todas as células com um kernel Python 3. A primeira instala apenas as dependências ausentes.
2. No painel ao final, clique em **Escolher imagem** e selecione um arquivo do seu computador.
3. Clique em **Gerar paleta** para ver a reconstrução e as três cores em **RGB e HEX**.
4. Para copiar os valores, selecione o texto na caixa de cores e pressione **Ctrl+C**. Para outra imagem, repita a seleção.

Mantenha o kernel ativo para usar os botões. O upload funciona pelo navegador, sem seletor nativo do Windows. Se a fotografia do trabalho estiver disponível, o botão **Usar foto do trabalho** permite testar sem upload.

O processamento mantém a proporção, limita o maior lado a 256 pixels e executa **um passo de gradiente com Goldstein**, como na Parte 1, com `tau = 0,005`. Não é uma otimização até convergir. Imagens com poucas cores podem repetir cores na paleta; imagens estacionárias são retornadas sem busca. Transparências são compostas sobre fundo branco.

In [1]:
# Instala somente os pacotes ainda ausentes no ambiente deste kernel.
import importlib.util
import subprocess
import sys

dependencias = {"numpy": "numpy", "PIL": "pillow", "matplotlib": "matplotlib", "ipywidgets": "ipywidgets"}
ausentes = [pacote for modulo, pacote in dependencias.items() if importlib.util.find_spec(modulo) is None]
if ausentes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *ausentes])
print("Dependências disponíveis. Execute as próximas células.")

Dependências disponíveis. Execute as próximas células.


## Preparação e cálculo

As funções abaixo são independentes de arquivos Python externos. A paleta é inicializada pelos percentis de intensidade e o objetivo/gradiente usam a mesma formulação suave da Parte 1. As componentes são arredondadas apenas para mostrar RGB de 8 bits e códigos hexadecimais.

In [2]:
from dataclasses import dataclass
from pathlib import Path
from io import BytesIO
import html

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from PIL import Image, ImageOps
from IPython.display import display, HTML

TAU = 0.005
LADO_MAXIMO = 256

@dataclass
class ResultadoPaleta:
    imagem: Image.Image
    reconstrucao: Image.Image
    paleta_inicial: np.ndarray
    paleta: np.ndarray
    custo_inicial: float
    custo_final: float
    alfa: float
    passos: int

    @property
    def rgb(self):
        # Conversão apenas para exibição; o cálculo usa ponto flutuante.
        return np.rint(255 * np.clip(self.paleta, 0, 1)).astype(np.uint8)

    @property
    def hexadecimais(self):
        return ["#" + "".join(f"{int(v):02X}" for v in cor) for cor in self.rgb]


def custo_e_gradiente(x, pixels, tau=0.005):
    """Objetivo suave e gradiente usados na Parte 1, com avaliação estável."""
    residuos = x.reshape(3, 3)[None, :, :] - pixels[:, None, :]
    distancias = np.sum(residuos ** 2, axis=2)
    minimo = distancias.min(axis=1, keepdims=True)
    exponenciais = np.exp(-(distancias - minimo) / tau)
    pesos = exponenciais / exponenciais.sum(axis=1, keepdims=True)
    custo = np.mean(minimo[:, 0] - tau * np.log(exponenciais.mean(axis=1)))
    gradiente = 2 * np.mean(pesos[:, :, None] * residuos, axis=0).ravel()
    return float(custo), gradiente


def inicializar_paleta(pixels):
    """Mesmos percentis da Parte 1, com tratamento de imagens com poucas cores."""
    ordem = np.argsort(pixels.mean(axis=1), kind="stable")
    indices = np.rint(np.array([0.1, 0.5, 0.9]) * (len(pixels) - 1)).astype(int)
    paleta = pixels[ordem[indices]].copy()
    if len(np.unique(paleta, axis=0)) == 3:
        return paleta

    # Completa cores repetidas com pixels distintos mais distantes dos já usados.
    unicas = np.unique(pixels, axis=0)
    escolhidas = [paleta[0]]
    while len(escolhidas) < min(3, len(unicas)):
        distancias = ((unicas[:, None, :] - np.array(escolhidas)[None, :, :]) ** 2).sum(axis=2)
        escolhidas.append(unicas[np.argmax(distancias.min(axis=1))])
    while len(escolhidas) < 3:
        escolhidas.append(escolhidas[-1].copy())
    return np.array(escolhidas)


def gerar_paleta(imagem, tau=0.005, lado_maximo=256):
    """Prepara uma imagem e executa até um passo de gradiente com Goldstein.

    Uma imagem estacionária é retornada sem busca. Os arquivos de entrada e
    os resultados do notebook não são modificados.
    """
    if not np.isfinite(tau) or tau <= 0 or lado_maximo < 1:
        raise ValueError("Tau e tamanho máximo devem ser positivos.")
    preparada = ImageOps.exif_transpose(imagem)
    if preparada.mode in ("RGBA", "LA") or "transparency" in preparada.info:
        rgba = preparada.convert("RGBA")
        fundo = Image.new("RGBA", rgba.size, "white")
        preparada = Image.alpha_composite(fundo, rgba)
    preparada = preparada.convert("RGB")
    preparada.thumbnail((lado_maximo, lado_maximo), Image.Resampling.LANCZOS)
    dados = np.asarray(preparada, dtype=np.float64) / 255
    pixels = dados.reshape(-1, 3)
    inicial = inicializar_paleta(pixels)
    x0 = inicial.ravel()
    f0, g0 = custo_e_gradiente(x0, pixels, tau)
    alfa, passos, final, f1 = 0.0, 0, x0.copy(), f0

    if np.linalg.norm(g0) > 1e-12:
        direcao = -g0
        derivada = float(g0 @ direcao)
        inferior, superior, tentativa = 0.0, None, 1.0
        for _ in range(100):
            candidato = x0 + tentativa * direcao
            valor, _ = custo_e_gradiente(candidato, pixels, tau)
            cota_inferior = f0 + 0.9 * tentativa * derivada
            cota_superior = f0 + 0.1 * tentativa * derivada
            if np.isfinite(valor) and cota_inferior <= valor <= cota_superior:
                alfa, passos, final, f1 = tentativa, 1, candidato, valor
                break
            if not np.isfinite(valor) or valor > cota_superior:
                superior = tentativa
            else:
                inferior = tentativa
            proxima = 2 * tentativa if superior is None else (inferior + superior) / 2
            if not np.isfinite(proxima) or proxima <= 0 or proxima == tentativa:
                raise RuntimeError("A busca não encontrou um passo válido.")
            tentativa = proxima
        else:
            raise RuntimeError("Goldstein não foi satisfeita em 100 tentativas.")

    paleta = final.reshape(3, 3)
    distancias = ((pixels[:, None, :] - paleta[None, :, :]) ** 2).sum(axis=2)
    reconstruida = paleta[np.argmin(distancias, axis=1)].reshape(dados.shape)
    rgb8 = np.rint(255 * np.clip(reconstruida, 0, 1)).astype(np.uint8)
    return ResultadoPaleta(preparada, Image.fromarray(rgb8), inicial, paleta, f0, f1, alfa, passos)

## Escolher e processar a imagem

O arquivo enviado é lido em memória. A ferramenta não sobrescreve a imagem original e não grava resultados nas pastas dos experimentos. Os valores de ponto flutuante ficam em `ultimo_resultado.paleta`, após uma execução bem-sucedida.

In [3]:
ultimo_resultado = None

def ler_upload(valor):
    """Lê tanto o formato de ipywidgets 8 quanto o formato da versão 7."""
    if not valor:
        raise ValueError("Escolha uma imagem antes de gerar a paleta.")
    if isinstance(valor, dict):
        nome, item = next(iter(valor.items()))
        nome = item.get("name", nome)
    else:
        item = valor[0]
        nome = item["name"]
    return str(nome), bytes(item["content"])

def exibir_resultado(resultado, nome):
    display(HTML("<h3>" + html.escape(nome) + "</h3>"))
    fig, eixos = plt.subplots(1, 2, figsize=(9, 5))
    for ax, imagem, titulo in zip(eixos,
            (resultado.imagem, resultado.reconstrucao),
            ("Imagem de trabalho", "Reconstrução com três cores")):
        ax.imshow(imagem)
        ax.set_title(titulo)
        ax.axis("off")
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    cartoes = []
    linhas = []
    for rgb, hexa in zip(resultado.rgb, resultado.hexadecimais):
        rotulo = f"RGB ({int(rgb[0])}, {int(rgb[1])}, {int(rgb[2])})"
        cartoes.append(f'<div style="flex:1;min-width:140px"><div style="background:{hexa};height:60px;border:1px solid #aaa"></div><p><b>{hexa}</b><br>{rotulo}</p></div>')
        linhas.append(f"{hexa} — {rotulo}")
    display(HTML('<div style="display:flex;gap:16px;flex-wrap:wrap">' + "".join(cartoes) + '</div>'))
    display(widgets.Textarea(value=chr(10).join(linhas), description="Copiar cores:",
        rows=3, layout=widgets.Layout(width="100%"), style={"description_width":"initial"}))
    if resultado.passos:
        reducao = 100 * (resultado.custo_inicial - resultado.custo_final) / resultado.custo_inicial
        print(f"Um passo concluído. Redução do objetivo: {reducao:.2f}%. Alfa: {resultado.alfa:g}.")
    else:
        print("Gradiente numericamente nulo: paleta retornada sem busca.")
    print(f"Resolução de trabalho: {resultado.imagem.width} × {resultado.imagem.height} pixels.")

def processar_conteudo(nome, conteudo):
    global ultimo_resultado
    ultimo_resultado = None
    with Image.open(BytesIO(conteudo)) as imagem:
        resultado = gerar_paleta(imagem, tau=TAU, lado_maximo=LADO_MAXIMO)
    ultimo_resultado = resultado
    exibir_resultado(resultado, nome)
    return resultado

def testar_arquivo(caminho):
    """Alternativa ao upload quando a imagem está acessível ao kernel."""
    arquivo = Path(caminho).expanduser()
    return processar_conteudo(arquivo.name, arquivo.read_bytes())

In [4]:
upload = widgets.FileUpload(accept="image/*", multiple=False, description="Escolher imagem")
botao_gerar = widgets.Button(description="Gerar paleta", button_style="primary", disabled=True)
botao_exemplo = widgets.Button(description="Usar foto do trabalho")
status = widgets.HTML("Escolha uma imagem para começar.")
saida = widgets.Output()

nome_exemplo = "nascer do sol rio de janeiro.jpeg"
exemplos = [Path(nome_exemplo), Path("programacao nao linear") / "trabalho 1" / nome_exemplo]
arquivo_exemplo = next((p for p in exemplos if p.is_file()), None)
botao_exemplo.disabled = arquivo_exemplo is None

def atualizar_selecao(change):
    botao_gerar.disabled = not bool(upload.value)
    if upload.value:
        nome, _ = ler_upload(upload.value)
        status.value = "Selecionado: " + html.escape(nome) + ". Clique em Gerar paleta."
    else:
        status.value = "Escolha uma imagem para começar."

def executar_interface(usar_exemplo=False):
    global ultimo_resultado
    botao_gerar.disabled = True
    botao_exemplo.disabled = True
    status.value = "Processando imagem…"
    saida.clear_output(wait=True)
    with saida:
        try:
            if usar_exemplo:
                if arquivo_exemplo is None:
                    raise FileNotFoundError("Envie uma imagem: a foto de exemplo não está nesta pasta.")
                nome, conteudo = arquivo_exemplo.name, arquivo_exemplo.read_bytes()
            else:
                nome, conteudo = ler_upload(upload.value)
            processar_conteudo(nome, conteudo)
            status.value = "Paleta pronta. Os valores RGB e HEX estão abaixo."
        except (OSError, ValueError, RuntimeError, Image.DecompressionBombError) as erro:
            ultimo_resultado = None
            status.value = "Não foi possível gerar a paleta. Escolha uma imagem válida."
            print(str(erro))
        finally:
            botao_gerar.disabled = not bool(upload.value)
            botao_exemplo.disabled = arquivo_exemplo is None

upload.observe(atualizar_selecao, names="value")
botao_gerar.on_click(lambda _: executar_interface())
botao_exemplo.on_click(lambda _: executar_interface(usar_exemplo=True))
display(widgets.VBox([widgets.HBox([upload, botao_gerar, botao_exemplo]), status, saida]))

### Alternativa: indicar o caminho do arquivo

Se preferir não usar o upload, remova o `#` da chamada abaixo, informe o caminho da imagem e execute apenas essa célula. O caminho precisa existir no computador em que o kernel está rodando.

In [5]:
# testar_arquivo("minha_foto.jpeg")
# No Windows, também é possível: testar_arquivo(r"C:\Fotos\minha_foto.png")